In [4]:
url = "jdbc:postgresql://my-postgres-container:5432/postgres"
user = "postgres"
password = "postgres"

In [2]:
pyspark = SparkSession.builder.config("spark.driver.extraClassPath", "/opt/spark/jars/postgresql-42.6.0.jar") \
        .config("spark.executor.extraClassPath", "/opt/spark/jars/postgresql-42.6.0.jar") \
        .appName("Jupyter").getOrCreate()

24/12/27 21:28:32 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [5]:
events_df = spark.read.format("jdbc")\
            .option("url",url)\
            .option("driver","org.postgresql.Driver")\
            .option("dbtable","drivers_data")\
            .option("user",user)\
            .option("password",password)\
            .option("dbtable","events").load()

In [7]:
events_df.columns

['url', 'referrer', 'user_id', 'device_id', 'host', 'event_time']

In [8]:
events_df.createOrReplaceTempView('events')

In [17]:
create_user_cumulated = """
CREATE TABLE IF NOT EXISTS homework.users_cumulated (
    user_id STRING,
    -- The list of dates in the past where the user was active
    dates_active Array<DATE>,
    -- Current data for the user
    date DATE
);
"""

In [16]:
pyspark.sql('show databases').show()

+---------+
|namespace|
+---------+
| homework|
+---------+



In [18]:
pyspark.sql(create_user_cumulated)

DataFrame[]

In [23]:
user_cumulated_query = """
WITH yesterday AS (
    SELECT
        *
    FROM homework.users_cumulated
    WHERE date = to_date('2022-12-31')
),
    today AS (
            SELECT
                  CAST(user_id AS STRING),
                  CAST(event_time AS DATE) as date_active
              FROM events
              WHERE
                  CAST(event_time AS DATE) = to_date('2023-01-01')
                    AND user_id IS NOT NULL
              GROUP BY user_id, CAST(event_time AS DATE)
    )
SELECT
    COALESCE(t.user_id, y.user_id) as user_id,
    CASE WHEN y.dates_active IS NULL
            THEN array(t.date_active)
        WHEN t.date_active IS NULL
            THEN y.dates_active
        ELSE concat(array(t.date_active), y.dates_active)
        END as dates_active,
    COALESCE(t.date_active, date_add(y.date, 1)) AS date
    FROM today t
    FULL OUTER JOIN yesterday y
    ON t.user_id = y.user_id
"""

In [28]:
df = pyspark.sql(user_cumulated_query)

In [26]:
pyspark.sql('select * from events').show()

+--------------------+--------+--------------------+--------------------+--------------------+--------------------+
|                 url|referrer|             user_id|           device_id|                host|          event_time|
+--------------------+--------+--------------------+--------------------+--------------------+--------------------+
|                   /|    NULL|                NULL|99067568206611800...| www.zachwilson.tech|2023-01-09 11:55:...|
|                   /|    NULL|16647130749494100...|99067568206611800...| www.zachwilson.tech|2023-01-09 20:10:...|
|                   /|    NULL|                NULL|12048389584711700...|admin.zachwilson....|2023-01-10 04:43:...|
|                   /|    NULL|                NULL|12048389584711700...| www.zachwilson.tech|2023-01-14 05:25:...|
|                   /|    NULL|                NULL|12048389584711700...| www.zachwilson.tech|2023-01-14 05:25:...|
|                   /|    NULL|                NULL|12048389584711700...

In [29]:
df

DataFrame[user_id: string, dates_active: array<date>, date: date]

In [30]:
df.show()

+--------------------+------------+----------+
|             user_id|dates_active|      date|
+--------------------+------------+----------+
|15049317011116400...|[2023-01-01]|2023-01-01|
|94468873453980500...|[2023-01-01]|2023-01-01|
|13431269135311700...|[2023-01-01]|2023-01-01|
|13527073786389900...|[2023-01-01]|2023-01-01|
|13394024080763900...|[2023-01-01]|2023-01-01|
|17375809595660200...|[2023-01-01]|2023-01-01|
|17602861391509800...|[2023-01-01]|2023-01-01|
|17996457346796300...|[2023-01-01]|2023-01-01|
|82331189944719850...|[2023-01-01]|2023-01-01|
|16899212529494500...|[2023-01-01]|2023-01-01|
|56416791558782860...|[2023-01-01]|2023-01-01|
|14156386081429200...|[2023-01-01]|2023-01-01|
|10060569187331700...|[2023-01-01]|2023-01-01|
|14839625512044600...|[2023-01-01]|2023-01-01|
|12718856434460700...|[2023-01-01]|2023-01-01|
|13547985275289400...|[2023-01-01]|2023-01-01|
|52893914368348070...|[2023-01-01]|2023-01-01|
|12615598167083900...|[2023-01-01]|2023-01-01|
|153331627826

In [31]:
import pandas as pd

In [32]:
pd.set_option('display.max_colwidth', None)

In [33]:
df.to_pandas()

AttributeError: 'DataFrame' object has no attribute 'to_pandas'

In [36]:
df.pandas_api()

/opt/spark/python/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


,user_id,dates_active,date
0,15049317011116400000.000000000000000000,[2023-01-01],2023-01-01
1,9446887345398050000.000000000000000000,[2023-01-01],2023-01-01
2,13431269135311700000.000000000000000000,[2023-01-01],2023-01-01
3,13527073786389900000.000000000000000000,[2023-01-01],2023-01-01
4,13394024080763900000.000000000000000000,[2023-01-01],2023-01-01
5,17375809595660200000.000000000000000000,[2023-01-01],2023-01-01
6,17602861391509800000.000000000000000000,[2023-01-01],2023-01-01
7,17996457346796300000.000000000000000000,[2023-01-01],2023-01-01
8,8233118994471985000.000000000000000000,[2023-01-01],2023-01-01
9,16899212529494500000.000000000000000000,[2023-01-01],2023-01-01


In [37]:
events_df.pandas_api()

,url,referrer,user_id,device_id,host,event_time
0,/,None,None,9906756820661180000.000000000000000000,www.zachwilson.tech,2023-01-09 11:55:28.032000
1,/,None,16647130749494100000.000000000000000000,9906756820661180000.000000000000000000,www.zachwilson.tech,2023-01-09 20:10:27.610000
2,/,None,None,12048389584711700000.000000000000000000,admin.zachwilson.tech,2023-01-10 04:43:49.204000
3,/,None,None,12048389584711700000.000000000000000000,www.zachwilson.tech,2023-01-14 05:25:03.497000
4,/,None,None,12048389584711700000.000000000000000000,www.zachwilson.tech,2023-01-14 05:25:04.305000
5,/,None,None,12048389584711700000.000000000000000000,www.zachwilson.tech,2023-01-14 05:44:51.848000
6,/?author=4,None,None,8300763538851936000.000000000000000000,www.zachwilson.tech,2023-01-22 10:29:17.735000
7,/?author=5,None,2854030062999680500.000000000000000000,8300763538851936000.000000000000000000,www.zachwilson.tech,2023-01-22 10:29:19.083000
8,/,None,7221550789269574000.000000000000000000,520733535420276540.000000000000000000,www.zachwilson.tech,2023-01-21 23:59:01.355000
9,/robots.txt,None,None,None,www.eczachly.com,2023-01-22 00:04:54.058000


In [ ]:
user_cumulated_query = """
WITH yesterday AS (
    SELECT
        *
    FROM homework.users_cumulated
    WHERE date = to_date('2022-12-31')
),
    today AS (
            SELECT
                  CAST(user_id AS STRING),
                  CAST(event_time AS DATE) as date_active
              FROM events
              WHERE
                  CAST(event_time AS DATE) = to_date('2023-01-01')
                    AND user_id IS NOT NULL
              GROUP BY user_id, CAST(event_time AS DATE)
    )
SELECT
    COALESCE(t.user_id, y.user_id) as user_id,
    CASE WHEN y.dates_active IS NULL
            THEN array(t.date_active)
        WHEN t.date_active IS NULL
            THEN y.dates_active
        ELSE concat(array(t.date_active), y.dates_active)
        END as dates_active,
    COALESCE(t.date_active, date_add(y.date, 1)) AS date
    FROM today t
    FULL OUTER JOIN yesterday y
    ON t.user_id = y.user_id
"""